In [ ]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 26.9 MB/s eta 0:00:00


In [ ]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE   = "/content/drive/MyDrive/arxiv-llm-project"      # adjust to your shared folder
INPUT_FILE   = f"{DRIVE_BASE}/data/processed/chunked_corpus.jsonl"
INDEX_FILE   = f"{DRIVE_BASE}/data/processed/faiss_index.bin"
META_FILE    = f"{DRIVE_BASE}/data/processed/chunk_metadata.jsonl"

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE   = 256

Mounted at /content/drive


In [ ]:
print("Loading chunked corpus...")
chunks = []
with open(INPUT_FILE, "r") as f:
    for line in f:
        chunks.append(json.loads(line.strip()))

print(f"  Total chunks loaded: {len(chunks):,}")

# Quick sanity check — print one chunk
print("\nSample chunk keys:", list(chunks[0].keys()))
print("Sample chunk_id  :", chunks[0].get("chunk_id", chunks[0].get("paper_id")))
print("Sample text[:200]:", chunks[0]["text"][:200])

Loading chunked corpus...
  Total chunks loaded: 76,443

Sample chunk keys: ['chunk_id', 'paper_id', 'chunk_index', 'total_chunks', 'title', 'url', 'year', 'category', 'abstract', 'text']
Sample chunk_id  : 2201.00095v1_chunk_0
Sample text[:200]: Computer Vision Based Parking Optimization
System
Siddharth Chandrasekaran, Jeffrey Matthew Reginald, Wei Wang, Ting Zhu
Department of Computer Science and Electrical Engineering
University of Marylan


In [ ]:
print("\nLoading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in chunks]

print(f"Embedding {len(texts):,} chunks in batches of {BATCH_SIZE}...")
all_embeddings = embedder.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # cosine similarity via dot-product on unit vectors
)

print(f"Embedding matrix shape: {all_embeddings.shape}")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 76,443 chunks in batches of 256...


Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Embedding matrix shape: (76443, 384)


In [ ]:
dim = all_embeddings.shape[1]   # 384 for MiniLM
print(f"\nBuilding FAISS IndexFlatL2 with dim={dim}...")

index = faiss.IndexFlatL2(dim)
index.add(all_embeddings.astype(np.float32))

print(f"  Vectors in index: {index.ntotal:,}")


Building FAISS IndexFlatL2 with dim=384...
  Vectors in index: 76,443


In [ ]:
print(f"\nSaving FAISS index → {INDEX_FILE}")
faiss.write_index(index, INDEX_FILE)

print(f"Saving chunk metadata → {META_FILE}")
with open(META_FILE, "w") as f:
    for i, chunk in enumerate(chunks):
        meta = {
            "faiss_id":  i,
            "chunk_id":  chunk.get("chunk_id", f"{chunk['paper_id']}_chunk_{i}"),
            "paper_id":  chunk["paper_id"],
            "title":     chunk.get("title", ""),
            "url":       chunk.get("url", ""),
            "year":      chunk.get("year", ""),
            "category":  chunk.get("category", ""),
            # Store first 500 chars of text for display in UI later
            "text_preview": chunk["text"][:500],
        }
        f.write(json.dumps(meta) + "\n")

print("Done saving.")


Saving FAISS index → /content/drive/MyDrive/arxiv-llm-project/data/processed/faiss_index.bin
Saving chunk metadata → /content/drive/MyDrive/arxiv-llm-project/data/processed/chunk_metadata.jsonl
Done saving.


In [ ]:
def retrieve(query: str, k: int = 5):
    """Embed a query and return top-k chunk metadata records."""
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    distances, indices = index.search(q_emb, k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        meta = chunks[idx]
        results.append({
            "rank":     rank + 1,
            "distance": float(dist),
            "chunk_id": meta.get("chunk_id", idx),
            "paper_id": meta["paper_id"],
            "title":    meta.get("title", "N/A"),
            "preview":  meta["body_text"][:300],
        })
    return results

# Run test queries
TEST_QUERIES = [
    "transformer attention mechanism self-supervised learning",
    "graph neural networks node classification",
    "reinforcement learning from human feedback RLHF",
]

print("\n── Smoke Test ──────────────────────────────────────────")
for q in TEST_QUERIES:
    print(f"\nQuery: {q!r}")
    hits = retrieve(q, k=3)
    for h in hits:
        print(f"  [{h['rank']}] dist={h['distance']:.4f} | {h['title'][:60]} | {h['preview'][:100]}...")

In [ ]:
print("\n── Reload verification ─────────────────────────────────")
index2 = faiss.read_index(INDEX_FILE)
print(f"  Reloaded index ntotal: {index2.ntotal:,} ✓")

meta_count = sum(1 for _ in open(META_FILE))
print(f"  Metadata records: {meta_count:,} ✓")

assert index2.ntotal == meta_count, "MISMATCH: index size != metadata count!"
print("  Counts match ✓")
print("\n✅ Milestone 2.4 complete — faiss_index.bin + chunk_metadata.jsonl saved to Drive")